# External mapper deep dive (marketing appendix; not Fig 4d)

This notebook complements `idtrack/reproducibility/experiments/comparison.ipynb` by generating **extra** figures/tables that help you market IDTrack.

## Rationale

External identifier mappers (BioMart, MyGene.info, g:Profiler, gget) are extremely useful, but they typically behave as **point-in-time services**.
This notebook quantifies two practical consequences on a small, safe demo set:

- **Outcome semantics**: how often queries become 1→0 / 1→1 / 1→n under different backends and targets.
- **Cross-method variability**: how often different backends disagree on the returned target set.

This is *not* an accuracy benchmark; it is an evidence pack for the claim that IDTrack’s snapshot-bounded, auditable semantics are qualitatively different.

## Outputs (optional marketing pool)

- `idtrack-manuscript/figures/fig_external_mapper_outcome_profiles.pdf`
- `idtrack-manuscript/figures/fig_external_mapper_agreement_heatmaps.pdf`
- `idtrack/docs/_notebooks/idtrack_cache/experiments/comparison/external_mapper_demo_outputs.csv`

## Caching

- Uses the same cache directory as `comparison.ipynb`.
- If caches are missing, the notebook will query external APIs once and write pickles under the shared cache.


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import sys

# Add experiments/src to sys.path
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    apply_rcparams,
    experiments_cache_dir,
    idtrack_cache_dir,
    load_rcparams,
    manuscript_figures_dir,
    write_pickle,
    read_pickle,
)

try:
    apply_rcparams(load_rcparams())
except Exception as e:  # noqa: S110
    print('Warning: could not apply shared rcParams:', e)

if sns is not None:
    sns.set_theme(style='whitegrid', context='paper')

plt.rcParams.update({'savefig.dpi': 300, 'figure.dpi': 140})

IDTRACK_LOCAL_REPO = idtrack_cache_dir(REPO_ROOT)
CACHE_DIR = experiments_cache_dir(REPO_ROOT, experiment='comparison')
MANUSCRIPT_FIGURES = manuscript_figures_dir(REPO_ROOT)

print('Repo root:', REPO_ROOT)
print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('CACHE_DIR:', CACHE_DIR)


In [ ]:
# -------------------- Configuration --------------------

METHODS = ['pybiomart', 'mygene', 'gprofiler', 'gget']

QUERY_IDS = [
    'ENSG00000139618',  # BRCA2
    'ENSG00000141510',  # TP53
    'ENSG00000157764',  # BRAF
    'ENSG00000121879',  # KRAS
    'ENSG00000171862',  # PTEN
    'ENSG00000136997',  # MYC
    'ENSG00000146648',  # EGFR
    'ENSG00000141510.18',  # TP53 (versioned)
    'ENSG_DOES_NOT_EXIST',  # negative control
]

INPUT_DB = 'ensembl_gene'
SPECIES = 'human'

TARGETS = [
    {'label': 'HGNC symbols', 'output_db': 'HGNC Symbol'},
    {'label': 'UniProt accessions', 'output_db': 'UniProtKB/Swiss-Prot'},
]

PYBIOMART_RELEASE = 107

print('N queries:', len(QUERY_IDS))
print('Targets:', [t['output_db'] for t in TARGETS])


In [ ]:
# -------------------- Compute / load cached external mappings --------------------

import idtrack._external_mappers as ext


def _safe_tag(s: str) -> str:
    return ''.join(c if c.isalnum() or c in {'-', '_'} else '_' for c in str(s))


def _cache_path(method: str, output_db: str) -> Path:
    tag = f"extmap_{method}_in{INPUT_DB}_out{_safe_tag(output_db)}_species{SPECIES}_pybiomart{PYBIOMART_RELEASE}.pickle"
    return CACHE_DIR / tag


def _normalize(df: pd.DataFrame | None) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=['input_id', 'output_id', 'mapping'])
    cols = [c for c in ['input_id', 'output_id', 'mapping', 'method', 'input_db', 'output_db', 'release_used'] if c in df.columns]
    return df[cols].copy()


results: dict[tuple[str, str], pd.DataFrame] = {}
for target in TARGETS:
    output_db = str(target['output_db'])
    for method in METHODS:
        p = _cache_path(method, output_db)
        if p.exists():
            results[(output_db, method)] = _normalize(read_pickle(p))
            continue

        kwargs = {
            'ids': QUERY_IDS,
            'input_db': INPUT_DB,
            'output_db': output_db,
            'method': method,
            'species': SPECIES,
            'chunk_size': 200,
            'pause': 0.1,
            'verbose': 2,
        }
        if method == 'pybiomart':
            kwargs['release_for_pybiomart'] = PYBIOMART_RELEASE

        df = ext.convert_ids(**kwargs)
        df = _normalize(df)
        write_pickle(df, p)
        results[(output_db, method)] = df

print('Cached result blocks:', len(results))


In [ ]:
# -------------------- Summaries (outcomes + per-query output sets) --------------------

def outputs_by_input(df: pd.DataFrame) -> dict[str, set[str]]:
    if df is None or df.empty:
        return {str(i): set() for i in QUERY_IDS}
    out: dict[str, set[str]] = {}
    for inp, sub in df.groupby('input_id'):
        vals = [v for v in sub['output_id'].tolist() if v is not None and str(v).strip() not in {'', 'nan', 'None', 'null'}]
        out[str(inp)] = set(map(str, vals))
    # ensure all inputs exist
    for i in QUERY_IDS:
        out.setdefault(str(i), set())
    return out


def outcome_counts(df: pd.DataFrame) -> dict[str, int]:
    if df is None or df.empty:
        return {'1→0': len(set(QUERY_IDS)), '1→1': 0, '1→n': 0}
    per = df.drop_duplicates('input_id')
    vc = per['mapping'].value_counts().to_dict()
    return {
        '1→0': int(vc.get('1:0', 0)),
        '1→1': int(vc.get('1:1', 0)),
        '1→n': int(vc.get('1:n', 0)),
    }


summary_rows = []
output_sets: dict[tuple[str, str], dict[str, set[str]]] = {}

for target in TARGETS:
    output_db = str(target['output_db'])
    for method in METHODS:
        df = results.get((output_db, method), pd.DataFrame())
        summary_rows.append({'output_db': output_db, 'method': method, **outcome_counts(df)})
        output_sets[(output_db, method)] = outputs_by_input(df)

summary = pd.DataFrame(summary_rows).sort_values(['output_db', 'method']).reset_index(drop=True)
summary


In [ ]:
# -------------------- Figure: outcome profiles for HGNC vs UniProt --------------------

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), constrained_layout=True)

for ax, target in zip(axes, TARGETS, strict=False):
    output_db = str(target['output_db'])
    sub = summary[summary['output_db'] == output_db].set_index('method')[['1→0', '1→1', '1→n']]
    sub = sub.div(sub.sum(axis=1), axis=0)
    sub.plot(
        kind='bar',
        stacked=True,
        ax=ax,
        color=[MANUSCRIPT_COLORS['1→0'], MANUSCRIPT_COLORS['1→1'], MANUSCRIPT_COLORS['1→n']],
    )
    ax.set_ylim(0, 1)
    ax.set_ylabel('Fraction of queries')
    ax.set_title(f'Outcome profile (target: {output_db})')
    ax.legend(['1→0', '1→1', '1→n'], loc='upper right')

out_fig = MANUSCRIPT_FIGURES / 'fig_external_mapper_outcome_profiles.pdf'
fig.savefig(out_fig, bbox_inches='tight')
print('Saved:', out_fig)


In [ ]:
# -------------------- Figure: cross-method agreement heatmaps --------------------

def jaccard(a: set[str], b: set[str]) -> float:
    denom = len(a | b)
    return (len(a & b) / denom) if denom else 1.0


fig2, axes2 = plt.subplots(1, 2, figsize=(12.5, 5.0), constrained_layout=True)

for ax, target in zip(axes2, TARGETS, strict=False):
    output_db = str(target['output_db'])

    # method x method: mean Jaccard across queries
    mat = pd.DataFrame(index=METHODS, columns=METHODS, dtype=float)
    for m1 in METHODS:
        for m2 in METHODS:
            s1 = output_sets[(output_db, m1)]
            s2 = output_sets[(output_db, m2)]
            vals = [jaccard(s1[str(q)], s2[str(q)]) for q in QUERY_IDS]
            mat.loc[m1, m2] = float(np.mean(vals))

    if sns is not None:
        sns.heatmap(mat, ax=ax, cmap='Blues', vmin=0, vmax=1, square=True, cbar=True)
    else:
        im = ax.imshow(mat.values, cmap='Blues', vmin=0, vmax=1)
        fig2.colorbar(im, ax=ax)
        ax.set_xticks(range(len(METHODS)))
        ax.set_yticks(range(len(METHODS)))
        ax.set_xticklabels(METHODS, rotation=30, ha='right')
        ax.set_yticklabels(METHODS)

    ax.set_title(f'Mean cross-method agreement (target: {output_db})')
    ax.set_xlabel('Method')
    ax.set_ylabel('Method')

out_fig2 = MANUSCRIPT_FIGURES / 'fig_external_mapper_agreement_heatmaps.pdf'
fig2.savefig(out_fig2, bbox_inches='tight')
print('Saved:', out_fig2)


In [ ]:
# -------------------- Export demo output table (CSV) --------------------

rows = []
for target in TARGETS:
    output_db = str(target['output_db'])
    for method in METHODS:
        s = output_sets[(output_db, method)]
        for q in QUERY_IDS:
            rows.append(
                {
                    'output_db': output_db,
                    'method': method,
                    'input_id': str(q),
                    'n_outputs': len(s[str(q)]),
                    'outputs': ';'.join(sorted(s[str(q)])[:30]),
                }
            )

demo = pd.DataFrame(rows)
out_csv = CACHE_DIR / 'external_mapper_demo_outputs.csv'
demo.to_csv(out_csv, index=False)
print('Wrote:', out_csv)

demo.head(10)
